# Ford GoBike System Data - Exploratory Analysis
## February 2019

**Dataset Source:** [Ford GoBike System Data](https://www.fordgobike.com/system-data)  
**Author:** Data Analysis Project  
**Date:** 2019

---

## Introduction

This dataset includes information about individual rides made in the Ford GoBike bike-sharing system covering the greater San Francisco Bay area during **February 2019**. The dataset contains **183,412 trips** with 16 features including trip duration, start/end times and locations, bike ID, user type, member demographics, and bike-share program participation.

### Questions I Want to Answer:
1. What is the distribution of trip durations? Are there differences between user types?
2. What are the peak usage hours and days of the week?
3. How does gender and age relate to trip patterns?
4. Which stations are most popular as start/end points?
5. Do subscribers and customers use the system differently (duration, timing, demographics)?
6. What is the relationship between age and trip duration?
7. How does gender, user type, and time of day interact to influence trip behavior?

## Preliminary Wrangling

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Set global plot styles
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully.')

In [ ]:
def load_and_wrangle(filepath):
    """
    Load the Ford GoBike CSV file and perform preliminary wrangling.
    
    Steps:
    - Parse datetime columns
    - Create derived features (duration_min, age, hour_of_day, day_of_week)
    - Filter out extreme outliers in duration and age
    - Drop rows with missing critical demographic data
    
    Parameters
    ----------
    filepath : str or Path
        Path to the CSV file.
    
    Returns
    -------
    pd.DataFrame
        Cleaned and enriched DataFrame.
    """
    df = pd.read_csv(filepath, on_bad_lines='skip')
    
    # Parse datetime columns
    df['start_time'] = pd.to_datetime(df['start_time'])
    df['end_time']   = pd.to_datetime(df['end_time'])
    
    # Create derived features
    df['duration_min']  = df['duration_sec'] / 60
    df['age']           = 2019 - df['member_birth_year']
    df['hour_of_day']   = df['start_time'].dt.hour
    df['day_of_week']   = df['start_time'].dt.day_name()
    df['day_num']       = df['start_time'].dt.dayofweek   # 0=Monday
    df['date']          = df['start_time'].dt.date
    
    # Filter unrealistic ages (keep 18-80)
    df = df[df['age'].between(18, 80) | df['age'].isna()]
    
    # Filter extreme duration outliers (> 3 hours = 180 min)
    df = df[df['duration_min'] <= 180]
    
    return df


def resolve_repo_paths():
    """Resolve data and figures paths whether the notebook is run from the repo root or notebooks/ folder."""
    base_candidates = [Path.cwd(), Path.cwd().parent]
    for base in base_candidates:
        data_file = base / 'data' / 'fordgobike_2019_02.csv'
        if data_file.exists():
            figures_dir = base / 'figures'
            figures_dir.mkdir(parents=True, exist_ok=True)
            return data_file, figures_dir
    raise FileNotFoundError('Could not find data/fordgobike_2019_02.csv from the current working directory.')


DATA_FILE, FIGURES_DIR = resolve_repo_paths()
df = load_and_wrangle(DATA_FILE)
print(f'Dataset shape after wrangling: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

In [ ]:
# Overview of data types and sample
print('Data Types:')
print(df.dtypes)
print('\nSample rows:')
df.head()

In [ ]:
# Missing values summary
print('Missing Values:')
missing = df.isnull().sum()
print(missing[missing > 0])
print(f'\nTotal rows: {len(df):,}')

### Summary of Wrangling

**What I kept:**
- All 16 original columns plus 6 engineered features (`duration_min`, `age`, `hour_of_day`, `day_of_week`, `day_num`, `date`).

**What I changed/removed:**
- Converted `start_time` and `end_time` to datetime objects.
- Filtered riders with ages outside 18–80 (removed implausible birth years).
- Removed trips longer than 3 hours (180 min) as extreme outliers that represent less than 0.5% of trips.
- ~8,265 rows are missing `member_birth_year` and `member_gender` — these belong to customers who did not register. I will retain them for non-demographic analyses.

**Features of interest:**
- **Primary:** `duration_min`, `user_type`, `member_gender`, `age`, `hour_of_day`, `day_of_week`
- **Supporting:** `start_station_name`, `bike_share_for_all_trip`

---
## Part 1: Univariate Exploration

**Question:** What are the individual distributions of key variables?  
I want to understand the shape and spread of trip durations, the breakdown of users by type and gender, and patterns in ride timing.

### 1.1 — Histogram: Trip Duration Distribution

In [ ]:
def plot_duration_histogram(df):
    """Plot histogram of trip durations with log-transformed x-axis."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Raw scale
    axes[0].hist(df['duration_min'], bins=50, color='steelblue', edgecolor='white')
    axes[0].axvline(df['duration_min'].mean(), color='red', linestyle='--', label=f'Mean: {df["duration_min"].mean():.1f} min')
    axes[0].axvline(df['duration_min'].median(), color='orange', linestyle='--', label=f'Median: {df["duration_min"].median():.1f} min')
    axes[0].set_xlabel('Trip Duration (minutes)')
    axes[0].set_ylabel('Number of Trips')
    axes[0].set_title('Trip Duration Distribution (Raw Scale)')
    axes[0].legend()
    
    # Log scale
    axes[1].hist(np.log10(df['duration_min']), bins=50, color='steelblue', edgecolor='white')
    axes[1].axvline(np.log10(df['duration_min'].mean()), color='red', linestyle='--', label=f'Mean: {df["duration_min"].mean():.1f} min')
    axes[1].axvline(np.log10(df['duration_min'].median()), color='orange', linestyle='--', label=f'Median: {df["duration_min"].median():.1f} min')
    tick_vals = [1, 2, 5, 10, 20, 60, 120]
    axes[1].set_xticks(np.log10(tick_vals))
    axes[1].set_xticklabels(tick_vals)
    axes[1].set_xlabel('Trip Duration (minutes, log scale)')
    axes[1].set_ylabel('Number of Trips')
    axes[1].set_title('Trip Duration Distribution (Log Scale)')
    axes[1].legend()
    
    plt.suptitle('Ford GoBike February 2019 — Trip Duration', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'hist_duration.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_duration_histogram(df)

**Observations:**  
The raw distribution is strongly right-skewed — most trips are short (5–15 min) but a small number of trips last over an hour. On the log scale, the distribution looks roughly unimodal and approximately normal, centered around ~8–9 minutes. The mean (12.1 min) is higher than the median (8.6 min), confirming the right skew. Log transformation will be used in downstream analyses involving duration.

### 1.2 — Bar Chart: User Type and Gender Distribution

In [ ]:
def plot_categorical_distributions(df):
    """Plot count plots for user_type, member_gender, and bike_share_for_all_trip."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # User type
    user_counts = df['user_type'].value_counts()
    bars0 = axes[0].bar(user_counts.index, user_counts.values, color=['#2196F3', '#FF9800'], edgecolor='white')
    for bar, val in zip(bars0, user_counts.values):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                     f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
    axes[0].set_title('User Type')
    axes[0].set_ylabel('Number of Trips')
    axes[0].set_ylim(0, user_counts.max() * 1.15)
    
    # Gender
    gender_counts = df['member_gender'].value_counts()
    bars1 = axes[1].bar(gender_counts.index, gender_counts.values, color=['#42A5F5', '#EF5350', '#66BB6A'], edgecolor='white')
    for bar, val in zip(bars1, gender_counts.values):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                     f'{val:,}\n({val/gender_counts.sum()*100:.1f}%)', ha='center', va='bottom', fontsize=10)
    axes[1].set_title('Member Gender')
    axes[1].set_ylabel('Number of Trips')
    axes[1].set_ylim(0, gender_counts.max() * 1.15)
    
    # Bike share for all
    bfa_counts = df['bike_share_for_all_trip'].value_counts()
    bars2 = axes[2].bar(bfa_counts.index, bfa_counts.values, color=['#78909C', '#26A69A'], edgecolor='white')
    for bar, val in zip(bars2, bfa_counts.values):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                     f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
    axes[2].set_title('Bike Share For All Program')
    axes[2].set_ylabel('Number of Trips')
    axes[2].set_ylim(0, bfa_counts.max() * 1.15)
    
    plt.suptitle('Categorical Variable Distributions', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'bar_categorical.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_categorical_distributions(df)

**Observations:**  
- **User type:** An overwhelming 89.1% of trips are made by **Subscribers** (annual members), while only 10.9% are made by casual **Customers**. This suggests the system is primarily used by commuters.
- **Gender:** Males account for 74.5% of trips, females 23.3%, and other genders 2.1%. The system has a significant gender imbalance.
- **Bike Share for All:** About 9.5% of trips are made under the discounted low-income equity program.

### 1.3 — Additional Univariate: Rides by Hour of Day

In [ ]:
def plot_hourly_distribution(df):
    """Plot the number of trip starts by hour of the day."""
    hourly = df['hour_of_day'].value_counts().sort_index()
    
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['#FF6B6B' if h in [8, 17] else '#4ECDC4' for h in hourly.index]
    bars = ax.bar(hourly.index, hourly.values, color=colors, edgecolor='white')
    
    ax.set_xticks(range(24))
    ax.set_xlabel('Hour of Day (0 = midnight)')
    ax.set_ylabel('Number of Trips')
    ax.set_title('Trip Starts by Hour of Day\n(Red = Morning & Evening Rush Hours)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    
    # Annotate peaks
    for h in [8, 17]:
        val = hourly[h]
        ax.annotate(f'{val:,}', xy=(h, val), xytext=(h, val + 500),
                    ha='center', fontweight='bold', color='#FF6B6B')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'bar_hourly.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_hourly_distribution(df)

**Observations:**  
There are two clear peaks at **8 AM** (21,056 trips) and **5 PM** (21,864 trips), confirming heavy commuter usage. Activity drops sharply after 7 PM and is minimal between midnight and 5 AM. This bimodal pattern is characteristic of a transit-oriented bike-sharing system.

---
## Part 2: Bivariate Exploration

**Question:** How do pairs of variables relate to each other? I want to explore how trip duration differs by user type and gender, whether age predicts duration, and how timing varies by user segment.

### 2.1 — Scatter Plot: Age vs. Trip Duration

In [ ]:
def plot_age_vs_duration(df):
    """Scatter plot of rider age vs log trip duration, with regression line."""
    clean = df.dropna(subset=['age']).copy()
    clean['log_dur'] = np.log10(clean['duration_min'])
    
    # Sample 5000 points for visibility
    sample = clean.sample(5000, random_state=42)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(sample['age'], sample['log_dur'], alpha=0.2, s=15, color='steelblue')
    
    # Add regression line
    m, b = np.polyfit(clean['age'], clean['log_dur'], 1)
    x_line = np.linspace(18, 80, 100)
    ax.plot(x_line, m*x_line + b, color='red', linewidth=2, label=f'Linear fit (r={clean["age"].corr(clean["log_dur"]):.3f})')
    
    tick_vals = [2, 5, 10, 20, 60, 120]
    ax.set_yticks(np.log10(tick_vals))
    ax.set_yticklabels(tick_vals)
    ax.set_xlabel('Rider Age (years)')
    ax.set_ylabel('Trip Duration (minutes, log scale)')
    ax.set_title('Rider Age vs. Trip Duration')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'scatter_age_duration.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    corr = clean['age'].corr(clean['duration_min'])
    print(f'Pearson r (age vs duration_min): {corr:.4f}')


plot_age_vs_duration(df)

**Observations:**  
There is virtually no linear correlation between rider age and trip duration (r ≈ 0.007). The scatter plot shows a uniform cloud across all ages. Trip duration appears to be driven more by purpose (commute vs. leisure) than by age.

### 2.2 — Box Plot: Trip Duration by User Type and Gender

In [ ]:
def plot_duration_boxplots(df):
    """Side-by-side box plots of trip duration by user type and gender."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # By user type
    groups = [df[df['user_type'] == t]['duration_min'].values for t in ['Subscriber', 'Customer']]
    bp = axes[0].boxplot(groups, labels=['Subscriber', 'Customer'],
                         patch_artist=True, notch=True,
                         boxprops=dict(facecolor='lightblue'),
                         medianprops=dict(color='red', linewidth=2))
    bp['boxes'][1].set_facecolor('lightyellow')
    
    # Annotate medians
    medians = [df[df['user_type'] == t]['duration_min'].median() for t in ['Subscriber', 'Customer']]
    for i, (med, label) in enumerate(zip(medians, ['Subscriber', 'Customer']), 1):
        axes[0].text(i + 0.1, med, f'{med:.1f} min', va='center', fontsize=10, color='darkred')
    
    axes[0].set_ylabel('Trip Duration (minutes)')
    axes[0].set_title('Trip Duration by User Type')
    axes[0].set_ylim(0, 80)
    
    # By gender
    gender_df = df.dropna(subset=['member_gender'])
    genders = ['Male', 'Female', 'Other']
    g_groups = [gender_df[gender_df['member_gender'] == g]['duration_min'].values for g in genders]
    colors = ['lightblue', 'lightpink', 'lightgreen']
    bp2 = axes[1].boxplot(g_groups, labels=genders, patch_artist=True, notch=True,
                          medianprops=dict(color='red', linewidth=2))
    for patch, color in zip(bp2['boxes'], colors):
        patch.set_facecolor(color)
    
    g_medians = [gender_df[gender_df['member_gender'] == g]['duration_min'].median() for g in genders]
    for i, med in enumerate(g_medians, 1):
        axes[1].text(i + 0.1, med, f'{med:.1f} min', va='center', fontsize=10, color='darkred')
    
    axes[1].set_ylabel('Trip Duration (minutes)')
    axes[1].set_title('Trip Duration by Gender')
    axes[1].set_ylim(0, 80)
    
    plt.suptitle('Trip Duration Distributions', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'box_duration.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_duration_boxplots(df)

**Observations:**  
- **User type:** Customers take significantly longer trips (median ~13.2 min) compared to Subscribers (median ~8.2 min). Customers also show more variability and a higher IQR. This suggests Customers use the bikes for leisure or tourism, while Subscribers use them for short, efficient commutes.
- **Gender:** Differences by gender are smaller. Females have a slightly higher median duration (9.5 min) than Males (8.2 min) and Others (9.3 min). All genders show similar spread.

### 2.3 — Bar Chart: Average Duration by Day of Week

In [ ]:
def plot_duration_by_day(df):
    """Bar chart of median trip duration by day of week."""
    day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    day_stats = df.groupby('day_of_week')['duration_min'].median().reindex(day_order)
    
    colors = ['#FF6B6B' if d in ['Saturday', 'Sunday'] else '#4ECDC4' for d in day_order]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(day_order, day_stats.values, color=colors, edgecolor='white')
    
    for bar, val in zip(bars, day_stats.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}', ha='center', fontsize=10)
    
    ax.set_xlabel('Day of Week')
    ax.set_ylabel('Median Trip Duration (minutes)')
    ax.set_title('Median Trip Duration by Day of Week\n(Red = Weekend)')
    ax.set_ylim(0, day_stats.max() * 1.15)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'bar_duration_day.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_duration_by_day(df)

**Observations:**  
Median trip duration is noticeably higher on **weekends** (Saturday and Sunday), reaching ~12–13 minutes versus ~8–9 minutes on weekdays. This reinforces the commuter vs. leisure split: weekday trips are short and purposeful, while weekend trips are longer and likely recreational.

### 2.4 — Heatmap: Trips by Hour and Day of Week

In [ ]:
def plot_heatmap_hour_day(df):
    """Heatmap of trip counts by hour of day and day of week."""
    day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    pivot = df.groupby(['day_of_week', 'hour_of_day']).size().unstack(fill_value=0)
    pivot = pivot.reindex(day_order)
    
    fig, ax = plt.subplots(figsize=(16, 6))
    sns.heatmap(pivot, ax=ax, cmap='YlOrRd', fmt='d', annot=False,
                linewidths=0.5, cbar_kws={'label': 'Number of Trips'})
    
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Day of Week')
    ax.set_title('Trip Count Heatmap: Day of Week × Hour of Day', fontsize=14, fontweight='bold')
    ax.set_xticklabels(range(24))
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'heatmap_hour_day.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_heatmap_hour_day(df)

**Observations:**  
The heatmap clearly reveals the **bimodal commuter pattern** on weekdays: dark red cells at 8 AM and 5 PM, Mon–Fri. On weekends, activity spreads more broadly across midday hours (10 AM–4 PM) with no sharp peaks. Tuesday and Wednesday appear to have the highest overall weekday ridership. Nights (11 PM–5 AM) are uniformly low throughout the week.

---
## Part 3: Multivariate Exploration

**Question:** How do three or more variables interact? I want to understand if subscriber vs. customer behavior is consistent across times and demographics, and how gender and age relate to usage patterns.

### 3.1 — Facet Plot: Trip Duration Distribution by User Type and Gender

In [ ]:
def plot_facet_duration(df):
    """FacetGrid of log trip duration by user type (columns) and gender (rows)."""
    subset = df.dropna(subset=['member_gender']).copy()
    subset['log_dur'] = np.log10(subset['duration_min'])
    
    g = sns.FacetGrid(subset, row='member_gender', col='user_type',
                      height=3, aspect=1.5, margin_titles=True)
    
    g.map(plt.hist, 'log_dur', bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    
    tick_vals = [2, 5, 10, 20, 60]
    for ax in g.axes.flat:
        ax.set_xticks(np.log10(tick_vals))
        ax.set_xticklabels(tick_vals)
        ax.set_xlabel('Duration (min)')
    
    g.set_axis_labels('Trip Duration (minutes, log scale)', 'Count')
    g.set_titles(row_template='Gender: {row_name}', col_template='User: {col_name}')
    g.figure.suptitle('Trip Duration Distribution by Gender & User Type', y=1.02, fontsize=13, fontweight='bold')
    
    plt.savefig(FIGURES_DIR / 'facet_duration_gender_usertype.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_facet_duration(df)

**Observations:**  
The facet grid shows that the **Subscriber distribution is consistently tight and short** regardless of gender — all three gender groups peak around 7–10 minutes. In contrast, **Customers have a broader, flatter distribution** extending into longer trips across all genders. Female customers tend to take slightly longer trips than male customers. The "Other" gender group shows the widest spread among customers.

### 3.2 — Facet Plot: Hourly Usage by User Type (Weekday vs Weekend)

In [ ]:
def plot_facet_hourly_usertype(df):
    """FacetGrid comparing hourly trip patterns for Subscriber vs Customer on weekdays vs weekends."""
    df2 = df.copy()
    df2['weekend'] = df2['day_of_week'].isin(['Saturday', 'Sunday']).map({True: 'Weekend', False: 'Weekday'})
    
    hourly = df2.groupby(['user_type', 'weekend', 'hour_of_day']).size().reset_index(name='trips')
    # Normalize by number of days
    day_counts = df2.groupby(['user_type', 'weekend'])['date'].nunique().reset_index(name='n_days')
    hourly = hourly.merge(day_counts, on=['user_type', 'weekend'])
    hourly['avg_trips'] = hourly['trips'] / hourly['n_days']
    
    g = sns.FacetGrid(hourly, col='user_type', row='weekend', height=3.5, aspect=1.6, margin_titles=True)
    g.map_dataframe(sns.barplot, x='hour_of_day', y='avg_trips', color='steelblue', order=range(24))
    g.set_axis_labels('Hour of Day', 'Avg Trips per Day')
    g.set_titles(row_template='{row_name}', col_template='{col_name}')
    g.figure.suptitle('Average Hourly Trips by User Type and Day Type', y=1.02, fontsize=13, fontweight='bold')
    
    plt.savefig(FIGURES_DIR / 'facet_hourly_usertype.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_facet_hourly_usertype(df)

**Observations:**  
This is the most revealing chart so far. **Subscribers on weekdays** show the classic double-peak commuter pattern at 8 AM and 5 PM. **Subscribers on weekends** shift to a leisurely midday pattern (10 AM–3 PM). **Customers** show a consistent single midday peak on both weekdays and weekends, suggesting they primarily use the system for tourism or exploration, regardless of day type. This strongly supports the hypothesis that subscribers are commuters and customers are leisure users.

### 3.3 — Scatter Plot with Multiple Encodings: Age, Duration, User Type, and Gender

In [ ]:
def plot_multivariate_scatter(df):
    """Scatter plot of age vs log duration, colored by user type and shaped by gender."""
    clean = df.dropna(subset=['age', 'member_gender']).copy()
    clean['log_dur'] = np.log10(clean['duration_min'])
    
    # Sample for clarity
    sample = clean.sample(4000, random_state=42)
    
    fig, ax = plt.subplots(figsize=(11, 7))
    
    colors = {'Subscriber': '#2196F3', 'Customer': '#FF9800'}
    markers = {'Male': 'o', 'Female': '^', 'Other': 's'}
    
    for utype in ['Subscriber', 'Customer']:
        for gender in ['Male', 'Female', 'Other']:
            mask = (sample['user_type'] == utype) & (sample['member_gender'] == gender)
            sub = sample[mask]
            if len(sub) == 0: continue
            ax.scatter(sub['age'], sub['log_dur'],
                       c=colors[utype], marker=markers[gender],
                       alpha=0.25, s=20,
                       label=f'{utype} / {gender}')
    
    tick_vals = [2, 5, 10, 20, 60, 120]
    ax.set_yticks(np.log10(tick_vals))
    ax.set_yticklabels(tick_vals)
    ax.set_xlabel('Rider Age (years)')
    ax.set_ylabel('Trip Duration (minutes, log scale)')
    ax.set_title('Age vs. Trip Duration\n(Color = User Type, Shape = Gender)', fontsize=13, fontweight='bold')
    
    # Custom legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0],[0], marker='o', color='w', markerfacecolor='#2196F3', markersize=8, label='Subscriber'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor='#FF9800', markersize=8, label='Customer'),
        Line2D([0],[0], marker='o', color='gray', markersize=8, label='Male'),
        Line2D([0],[0], marker='^', color='gray', markersize=8, label='Female'),
        Line2D([0],[0], marker='s', color='gray', markersize=8, label='Other'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', framealpha=0.8)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'scatter_multivariate.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_multivariate_scatter(df)

**Observations:**  
The multi-encoding scatter plot confirms that **user type (color) has a stronger effect on trip duration than gender (shape)**. Orange customer points cluster in the upper portion of the chart (longer trips) compared to blue subscriber points across all ages and genders. Female customers (orange triangles) show slightly longer durations than male customers on average. Age, regardless of user type or gender, remains weakly associated with trip duration.

---
## Conclusions

### Main Findings

1. **Trip duration is right-skewed** with a median of ~8.6 minutes (after filtering extremes). Log transformation normalizes the distribution well.

2. **The system is dominated by commuters:** 89.1% of trips are by annual Subscribers, and weekday ridership shows classic 8 AM/5 PM peaks.

3. **User type is the strongest predictor of trip behavior:** Subscribers take shorter, more purposeful trips; Customers take longer, more variable leisure trips regardless of day or time.

4. **Subscribers change their behavior on weekends,** shifting from commute peaks to a midday pattern — but their trips remain shorter than customers' trips.

5. **Gender differences exist but are moderate:** Female riders take slightly longer trips than males (median 9.5 vs 8.2 min), and male riders dominate (74.5%).

6. **Age has negligible correlation with trip duration** — the rider demographic (subscriber vs. customer) matters far more.

### Reflection

The most surprising finding was that age had almost no relationship with trip duration — I expected older riders to take shorter trips. The clearest story in this dataset is the **Subscriber vs. Customer behavioral divide**, which manifests in duration, timing, and day-of-week patterns. Future analysis could incorporate geographic data (start/end stations) to see whether customers and subscribers also use different parts of the network.